In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re



In [81]:
name = 'TEST037_NO_INTERR.csv'
experiment = 'ECG' #ECG or EEG

if experiment == 'ECG 2':
    num_CH = 2
elif experiment == 'ECG':
    num_CH = 1
elif experiment == 'EEG':
    num_CH = 4
CH_first = 0
v_max = 0.6

In [82]:
df = pd.read_csv(name, sep=',', dtype=str)
df.columns

Index(['SPI', ' Time', ' MISO', ' MOSI '], dtype='object')

In [83]:
print(df.head(5))

  SPI        Time  MISO  MOSI 
0   1  -9.79813ms  0x00   0x30
1   2  -9.79589ms  0x00   0x1E
2   3  -9.79364ms  0x00   0x65
3   4  -9.73476ms  0x00   0x30
4   5  -9.73252ms  0x00   0x20


In [84]:
df = df.drop(columns = ' MISO')
df = df.drop(columns = 'SPI')

print(df.head(5))

         Time MOSI 
0  -9.79813ms  0x30
1  -9.79589ms  0x1E
2  -9.79364ms  0x65
3  -9.73476ms  0x30
4  -9.73252ms  0x20


In [85]:
df.columns = ['Time', 'MOSI']

print(df.head(21))


          Time  MOSI
0   -9.79813ms  0x30
1   -9.79589ms  0x1E
2   -9.79364ms  0x65
3   -9.73476ms  0x30
4   -9.73252ms  0x20
5   -9.73026ms  0x9E
6   -9.67060ms  0x30
7   -9.66836ms  0x23
8   -9.66609ms  0x1F
9   -9.60637ms  0x30
10  -9.60412ms  0x26
11  -9.60187ms  0x4C
12  -9.54236ms  0x30
13  -9.54012ms  0x29
14  -9.53788ms  0xA9
15  -9.47822ms  0x30
16  -9.47596ms  0x2D
17  -9.47371ms  0x25
18  -9.41394ms  0x30
19  -9.41169ms  0x31
20  -9.40943ms  0x2E


In [86]:
df['MOSI'] = df['MOSI'].apply(lambda x: int(x, 16))

In [87]:
# Verifica cómo quedó
print(df.head(25))


          Time  MOSI
0   -9.79813ms    48
1   -9.79589ms    30
2   -9.79364ms   101
3   -9.73476ms    48
4   -9.73252ms    32
5   -9.73026ms   158
6   -9.67060ms    48
7   -9.66836ms    35
8   -9.66609ms    31
9   -9.60637ms    48
10  -9.60412ms    38
11  -9.60187ms    76
12  -9.54236ms    48
13  -9.54012ms    41
14  -9.53788ms   169
15  -9.47822ms    48
16  -9.47596ms    45
17  -9.47371ms    37
18  -9.41394ms    48
19  -9.41169ms    49
20  -9.40943ms    46
21  -9.35056ms    48
22  -9.34832ms    52
23  -9.34605ms   245
24  -9.28636ms    48


In [88]:
# Crea un diccionario para mapear los valores
mapeo = {
    48+CH_first: CH_first
}

# Comienza después de la última clave
ultima_clave = max(mapeo.keys())
ultimo_valor = max(mapeo.values())

for i in range(1, 5):
    nueva_clave = ultima_clave + i
    nuevo_valor = ultimo_valor + i
    mapeo[nueva_clave] = nuevo_valor

print(mapeo)

{48: 0, 49: 1, 50: 2, 51: 3, 52: 4}


In [89]:
# Aplica el mapeo y pon 'S' en el resto
df['type'] = df['MOSI'].astype(int).map(mapeo).fillna('S')
df = df.reset_index(drop=True)
print(df.head(40))


          Time  MOSI type
0   -9.79813ms    48  0.0
1   -9.79589ms    30    S
2   -9.79364ms   101    S
3   -9.73476ms    48  0.0
4   -9.73252ms    32    S
5   -9.73026ms   158    S
6   -9.67060ms    48  0.0
7   -9.66836ms    35    S
8   -9.66609ms    31    S
9   -9.60637ms    48  0.0
10  -9.60412ms    38    S
11  -9.60187ms    76    S
12  -9.54236ms    48  0.0
13  -9.54012ms    41    S
14  -9.53788ms   169    S
15  -9.47822ms    48  0.0
16  -9.47596ms    45    S
17  -9.47371ms    37    S
18  -9.41394ms    48  0.0
19  -9.41169ms    49  1.0
20  -9.40943ms    46    S
21  -9.35056ms    48  0.0
22  -9.34832ms    52  4.0
23  -9.34605ms   245    S
24  -9.28636ms    48  0.0
25  -9.28411ms    56    S
26  -9.28184ms   163    S
27  -9.22216ms    48  0.0
28  -9.21991ms    61    S
29  -9.21766ms    79    S
30  -9.15804ms    48  0.0
31  -9.15578ms    65    S
32  -9.15353ms   224    S
33  -9.09388ms    48  0.0
34  -9.09163ms    70    S
35  -9.08937ms    88    S
36  -9.02964ms    48  0.0
37  -9.02740

In [90]:
# Inicializamos idx como None
idx = None
# Recorremos los índices donde 'type' es distinto de 'S'
for i in df.index[df['type'] != 'S']:
    # Verificamos que haya al menos dos filas siguientes
    if (i + 2) < len(df):
        # Comprobamos que las dos siguientes filas tengan 'S'
        if (df.loc[i + 1, 'type'] == 'S') and (df.loc[i + 2, 'type'] == 'S'):
            idx = i
            break
    else:
        # Si no hay suficientes filas para verificar, no es un punto válido
        continue

# Si encontramos un índice válido, cortamos el dataframe
if idx is not None:
    df = df.loc[idx:].reset_index(drop=True)
else:
    # Si no se encontró un punto válido, el dataframe queda vacío o como prefieras manejarlo
    df = df.iloc[0:0].reset_index(drop=True)


In [91]:
print(df['type'][0])
print(df['type'][1])
print(df['type'][2])
print(df['type'][3])


0.0
S
S
0.0


In [92]:
# Diccionario para almacenar los arrays
resultados = {CH_first: []}
resultados_tiempos = {CH_first: []}



# Obtener el valor máximo actual de clave
ultima_clave = max(resultados.keys())

# Agregar N nuevas claves a ambos diccionarios
for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    resultados[nueva_clave] = []
    resultados_tiempos[nueva_clave] = []

# Iterar sobre el dataframe
for i, row in df.iterrows():
    tipo = row['type']
    tipo_tiempo = row['type']  
    if tipo in resultados:
        # Tomar las dos siguientes filas si existen
        sub_df = df.iloc[i+1:i+3]['MOSI']
        sub_df_times = df.iloc[i+1:i+3]['Time']
        # Guardar como array (puedes ajustar qué columnas guardar)
        resultados[tipo].append(sub_df.to_numpy())
        resultados_tiempos[tipo_tiempo].append(sub_df_times.to_numpy())
        




In [93]:

# Opcional: convertir las listas en arrays grandes (si quieres)
import numpy as np
for k in resultados:
    resultados[k] = np.concatenate(resultados[k])
    resultados_tiempos[k] = np.concatenate(resultados_tiempos[k])

# Ahora resultados[0], resultados[1], ... tienen los arrays deseados

In [94]:
for i in range (num_CH):
    print(len(resultados[i]))


628


In [95]:
# Creamos un nuevo diccionario con las filas pares eliminadas
resultados_filtrados = {}

for k, arr in resultados_tiempos.items():
    # Tomar los elementos en posiciones impares: 1, 3, 5, ...
    resultados_tiempos[k] = arr[1::2]


In [96]:
new_arr0 = []
new_arr1 = []
new_arr2 = []
new_arr3 = []
new_arr4 = []

if experiment == 'ECG':
    arr0 = resultados[0].flatten()
    for i in range(0, len(arr0)-1, 2):
        combined = arr0[i] * 256 + arr0[i+1]
        new_arr0.append(combined)
    new_arr0 = np.array(new_arr0)
elif experiment == 'ECG 2':
    arr0 = resultados[0].flatten()
    for i in range(0, len(arr0)-1, 2):
        combined = arr0[i] * 256 + arr0[i+1]
        new_arr0.append(combined)
    new_arr0 = np.array(new_arr0)
    
    arr1 = resultados[1].flatten()

    for i in range(0, len(arr1)-1, 2):
        combined = arr1[i] * 256 + arr1[i+1]
        new_arr1.append(combined)
    new_arr1 = np.array(new_arr1)

elif experiment == 'ECG':
    arr0 = resultados[0].flatten()
    for i in range(0, len(arr0)-1, 2):
        combined = arr0[i] * 256 + arr0[i+1]
        new_arr0.append(combined)
    new_arr0 = np.array(new_arr0)

    arr1 = resultados[1].flatten()

    for i in range(0, len(arr1)-1, 2):
        combined = arr1[i] * 256 + arr1[i+1]
        new_arr1.append(combined)
    new_arr1 = np.array(new_arr1)

    arr2 = resultados[2].flatten()
    for i in range(0, len(arr2)-1, 2):
        combined = arr2[i] * 256 + arr2[i+1]
        new_arr2.append(combined)
    new_arr2 = np.array(new_arr2)

    arr3 = resultados[3].flatten()
    for i in range(0, len(arr3)-1, 2):
        combined = arr3[i] * 256 + arr3[i+1]
        new_arr3.append(combined)
    new_arr3 = np.array(new_arr3)


In [97]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()


In [101]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()


In [99]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[0], y=new_arr0, mode='lines', name='Señal 0'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified',
    yaxis=dict(range=[0, 65536])
)

fig.show()

In [100]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[1], y=new_arr1, mode='lines', name='Señal 1'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
    hovermode='x unified',
    yaxis=dict(range=[0, 65536])
)

fig.show()

KeyError: 1

In [ ]:
if experiment == 'EEG':
    fig = go.Figure()


    # Agregamos cada señal como un trazo
    fig.add_trace(go.Scatter(x = resultados_tiempos[2], y=new_arr2, mode='lines', name='Señal 2'))

    # Opciones de layout
    fig.update_layout(
        title='ECG',
        xaxis_title='Tiempo',
        yaxis_title='Valor',
        hovermode='x unified',
        yaxis=dict(range=[0, 65536])
    )

    fig.show()

In [ ]:
if experiment == 'EEG':

    fig = go.Figure()


    # Agregamos cada señal como un trazo
    fig.add_trace(go.Scatter(x = resultados_tiempos[3], y=new_arr3, mode='lines', name='Señal 3'))

    # Opciones de layout
    fig.update_layout(
        title='Señales muestreadas superpuestas',
        xaxis_title='Tiempo',
        yaxis_title='Valor',
        hovermode='x unified',
        yaxis=dict(range=[0, 65536])
    )

    fig.show()

In [ ]:
if experiment == 'EEG':

  fig = go.Figure()


  # Agregamos cada señal como un trazo
  fig.add_trace(go.Scatter(x = resultados_tiempos[4], y=new_arr4, mode='lines', name='Señal 4'))

  # Opciones de layout
  fig.update_layout(
      title='BATT level',
      xaxis_title='Tiempo',
      yaxis_title='Valor',
      hovermode='x unified',
    yaxis=dict(range=[0, 65536])
  )

  fig.show()

In [ ]:
if experiment == 'EEG':
    ceros = np.zeros(100)  
    # FFT
    X = np.fft.fft(np.concatenate((ceros,(new_arr1-32768))))

    # Número de muestras
    N = len(X)

    # Frecuencias asociadas (eje x)
    freqs = np.fft.fftfreq(N, 1/f)

    # Magnitud (módulo)
    magnitud = np.abs(X)

    # Para mostrar solo la mitad positiva (frecuencias positivas)
    idxs = freqs >= 0

    plt.plot(freqs[idxs], magnitud[idxs])
    plt.xlabel("Frecuencia (Hz)")
    plt.ylabel("Magnitud")
    plt.title("Espectro de la señal")
    plt.show()
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x = freqs[idxs],
        y=magnitud[idxs],
        mode='lines',
        name='Valores concatenados'
    ))

    fig.update_layout(
        title='FFT',
        xaxis_title='Frecuencia',
        yaxis_title='Magnitud',
        hovermode='x unified'
    )

    fig.show()